In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from ariel_pred.dataset import DataLoaderAndCalibrator
from ariel_pred.preprocessing import SGSmoothing
from ariel_pred.plots import plot_white_curve
from ariel_pred.transit import FunctionFittingBasedPhaseDetector
from ariel_pred.models import TransitMultiplicationFactorFinder, SValuesCNN, SValuesCNNTrainer
from ariel_pred.features import WavelengthsGroupsMultiplierFinder
from ariel_pred.metrics import score, gll
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import random
from scipy.optimize import minimize
from tqdm.auto import tqdm
import pandas as pd
import torch
from torch import nn
import scipy

In [4]:
data_loader = DataLoaderAndCalibrator(
    data_path=Path("../data/raw"),
    output_path=Path("../data/calibrated/full"),
    force_recalibration=False,
    cut_airs_channels=True,
    binning=4,
    n_jobs=4
)
train_data, train_labels = data_loader.load_all_train_data()
train_data.shape, train_labels.shape

Loading calibrated train data...


((1100, 1406, 283), (1100, 283))

In [5]:
features_extractor = WavelengthsGroupsMultiplierFinder()

features = features_extractor.extract_features(
    train_data,
)

features.shape

100%|██████████| 1100/1100 [03:24<00:00,  5.37it/s]


(1100, 283)

In [6]:
np.mean((0.945*features - train_labels)**2)**0.5

np.float64(0.0007461554133075029)

In [7]:
gll(np.concatenate([0.945*features, np.ones_like(features)*0.000715], axis=1), train_labels)

0.33588261161816463

In [8]:
features.min(), features.max()

(np.float64(0.0006007662495559545), np.float64(0.09456829514552487))